# 02_preparacao_produtiva_territorio

Preparacao da base UFPA produtiva/territorio para modelagem de ML (PU Learning).


## 1. Introdução

Esta base reúne informações produtivas e territoriais do CAF, em escala administrativa (milhões de registros).

O objetivo é gerar um dataset limpo, consistente e eficiente para modelagem de ML, com foco em:
- limpeza essencial;
- criação de variáveis derivadas;
- padronização de tipos;
- redução de memória;
- persistência final em parquet.

## 2. Imports e Configuração do Ambiente

Importamos bibliotecas essenciais e definimos caminhos de entrada e saída.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
from IPython.display import display

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

DEFAULT_INPUT_PATH = (
    ROOT
    / 'dados'
    / 'dados_brutos'
    / 'a_tcc_v2_ufpa_produtiva_territorio_compilado.parquet'
)
INPUT_PATH = Path(os.getenv('PRODUTIVA_TERRITORIO_PATH', str(DEFAULT_INPUT_PATH)))
OUTPUT_PATH = (
    ROOT
    / 'dados'
    / 'dados_processados'
    / 'df_caf_ufpa_produtiva_territorio.parquet'
)

if not INPUT_PATH.exists():
    raise FileNotFoundError(f'Arquivo não encontrado: {INPUT_PATH}')

print(f'ROOT: {ROOT}')
print(f'INPUT_PATH: {INPUT_PATH}')
print(f'OUTPUT_PATH: {OUTPUT_PATH}')
print(f'pyarrow: {pa.__version__}')

## 3. Carregamento do Dataset

Carregamos o parquet bruto e inspecionamos rapidamente estrutura, tamanho e tipos.

In [ ]:
colunas_interesse = [
    'id_familia',
    # 'ibge',
    'cd_uf',
    'id_regiao',
    'nr_modulo_fiscal',
    'lat_imovel_principal',
    'lon_imovel_principal',
    'n_imoveis_total',
    'n_imoveis_ha',
    'n_imoveis_m2',
    'n_imoveis_m3',
    'soma_area_ha',
    'soma_area_m2',
    'soma_volume_m3',
    'tem_incra',
    # 'qtd_imoveis_incra',
    # 'qtd_imoveis_com_cd_incra',
]

df = pd.read_parquet(INPUT_PATH, engine='pyarrow', columns=colunas_interesse)

memoria_inicial_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)

print(f'Shape inicial: {df.shape}')
print(f'Memória inicial: {memoria_inicial_mb:,.2f} MB')
display(df.head(3))
df.info(show_counts=True)

## 4. Diagnóstico Inicial dos Dados

Nesta etapa fazemos checagens leves de missing, cardinalidade e distribuições principais.

In [ ]:
print('Missing por coluna (ordem decrescente):')
display(df.isna().sum().sort_values(ascending=False))

print('Cardinalidade (nunique) por coluna:')
display(df.nunique(dropna=True).sort_values())

for col in ['cd_uf', 'id_regiao', 'tem_incra', 'n_imoveis_total']:
    if col in df.columns:
        print(f'\nDistribuição de {col}:')
        display(df[col].value_counts(dropna=False).head(15))

if 'id_familia' in df.columns:
    duplicados_id_familia = int(df.duplicated(subset=['id_familia']).sum())
    print(f'\nDuplicatas de id_familia: {duplicados_id_familia:,}')

## 5. Limpeza Básica

Aplicamos limpeza essencial para manter consistência sem remover registros desnecessariamente:
- remoção de duplicatas por `id_familia`;
- ajuste de coordenadas inválidas (fora da faixa geográfica do Brasil) para `NaN`;
- criação da flag `st_geo_valida`.

In [ ]:
linhas_antes_limpeza = len(df)

# 5.1) Remoção de duplicatas por chave administrativa (id_familia)
if 'id_familia' in df.columns:
    n_dup = int(df.duplicated(subset=['id_familia']).sum())
    if n_dup > 0:
        df = df.drop_duplicates(subset=['id_familia'], keep='first').copy()
    print(f'Duplicatas removidas por id_familia: {n_dup:,}')

# 5.2) Tratamento de coordenadas inválidas
lat_col = 'lat_imovel_principal'
lon_col = 'lon_imovel_principal'

if lat_col in df.columns and lon_col in df.columns:
    # Faixa plausível de coordenadas para o Brasil
    lat_min, lat_max = -33.75, 5.27
    lon_min, lon_max = -73.99, -28.84

    lat = df[lat_col]
    lon = df[lon_col]

    coord_presente = lat.notna() & lon.notna()
    coord_valida = (
        coord_presente
        & lat.between(lat_min, lat_max)
        & lon.between(lon_min, lon_max)
    )

    n_coord_invalidas = int((coord_presente & ~coord_valida).sum())

    # Mantém o registro, mas marca a coordenada inválida como ausente
    df.loc[coord_presente & ~coord_valida, [lat_col, lon_col]] = np.nan
    df['st_geo_valida'] = coord_valida.astype('int8')

    print(f'Registros com coordenada inválida tratados como NaN: {n_coord_invalidas:,}')
else:
    df['st_geo_valida'] = np.int8(0)

print(f'Shape antes da limpeza: ({linhas_antes_limpeza:,}, {len(colunas_interesse)})')
print(f'Shape após limpeza: ({df.shape[0]:,}, {df.shape[1]})')

print('\nChecagem rápida de coordenadas:')
print(f"Total linhas: {len(df):,}")
print(f"Missing latitude: {int(df['lat_imovel_principal'].isna().sum()):,}")
print(f"Missing longitude: {int(df['lon_imovel_principal'].isna().sum()):,}")
print('Geo válida vs lat missing:')
print(
    df.groupby('st_geo_valida')['lat_imovel_principal']
    .apply(lambda x: x.isna().sum())
)

## 6. Padronização de Tipos de Dados

In [ ]:
# 6.1) Identificadores em string
for col in ['id_familia', 'ibge']:
    if col in df.columns:
        df[col] = df[col].astype('string')

# 6.2) Códigos e flags em int8
# cd_uf deve ser numérico; convertemos explicitamente antes do cast final.
if 'cd_uf' in df.columns:
    df['cd_uf'] = pd.to_numeric(df['cd_uf'], errors='coerce')
    if df['cd_uf'].isna().any():
        raise ValueError('cd_uf contém valores não numéricos ou nulos após conversão.')
    df['cd_uf'] = df['cd_uf'].astype('int8')

int8_cols = [
    'id_regiao',
    'nr_modulo_fiscal',
    'n_imoveis_total',
    'n_imoveis_ha',
    'n_imoveis_m2',
    'n_imoveis_m3',
    'tem_incra',
    'st_geo_valida',
]

for col in int8_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        if df[col].isna().any():
            raise ValueError(f'{col} contém valores inválidos para cast em int8.')
        df[col] = df[col].astype('int8')

# 6.3) Variáveis contínuas em float32
for col in [
    'lat_imovel_principal',
    'lon_imovel_principal',
    'soma_area_ha',
    'soma_area_m2',
    'soma_volume_m3',
]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('float32')

print('Dtypes após padronização:')
display(df.dtypes)

## 7. Tratamento de Missing

Aplicamos preenchimento apenas em colunas textuais, preservando a consistência dos tipos numéricos.

Observação: `cd_uf` permanece numérica (`int8`) e não recebe `'Não informado'`.

In [ ]:
# Preenchimento conservador apenas para colunas string
for col in ['id_familia', 'ibge']:
    if col in df.columns and pd.api.types.is_string_dtype(df[col]):
        if df[col].isna().any():
            df[col] = df[col].fillna('Não informado')

print('Missing após tratamento (top 10):')
display(df.isna().sum().sort_values(ascending=False).head(10))

## 8. Otimização de Memória

Como os casts foram feitos de forma explêcita no passo anterior, não aplicamos downcast automático adicional.

Aqui medimos apenas o ganho de memória alcançado.

In [ ]:
memoria_final_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)
reducao_pct = ((memoria_inicial_mb - memoria_final_mb) / memoria_inicial_mb) * 100

print(f'Memória inicial: {memoria_inicial_mb:,.2f} MB')
print(f'Memória final:   {memoria_final_mb:,.2f} MB')
print(f'Redução:         {reducao_pct:,.2f}%')

## 9. Validação Final

Verificamos estrutura final, coerência básica e possíveis valores impossíveis restantes.

In [ ]:
print(f'Shape final (pré-reset): {df.shape}')

df.info(show_counts=True)
display(df.head(5))


In [ ]:

# Coerência de contagens
if {
    'n_imoveis_total',
    'n_imoveis_ha',
    'n_imoveis_m2',
    'n_imoveis_m3',
}.issubset(df.columns):
    inconsist_total = int(
        (
            df['n_imoveis_total']
            != (df['n_imoveis_ha'] + df['n_imoveis_m2'] + df['n_imoveis_m3'])
        ).sum()
    )
    print(
        'Inconsistências n_imoveis_total vs soma de componentes: '
        f'{inconsist_total:,}'
    )


In [ ]:
# Checagem de coordenadas fora da faixa, ignorando NaN
if {'lat_imovel_principal', 'lon_imovel_principal'}.issubset(df.columns):
    lat_ok = (
        df['lat_imovel_principal'].isna()
        | df['lat_imovel_principal'].between(-33.75, 5.27)
    )
    lon_ok = (
        df['lon_imovel_principal'].isna()
        | df['lon_imovel_principal'].between(-73.99, -28.84)
    )
    inconsist_geo = int((~(lat_ok & lon_ok)).sum())
    print(f'Registros com coordenada fora da faixa após limpeza: {inconsist_geo:,}')

## 10. Reset do Índice

Após remoções e filtros, resetamos o índice para manter continuidade e evitar índices quebrados.

In [ ]:
df = df.reset_index(drop=True)
print(f'Shape final (pós-reset): {df.shape}')
print(f'índice contínuo: {df.index.min()} até {df.index.max()}')

## 11. Persistência do Dataset

Salvamos a versão limpa em parquet para uso nas próximas etapas do pipeline.

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(OUTPUT_PATH, index=False)

print(f'Arquivo salvo em: {OUTPUT_PATH}')
print(f'Shape final salvo: {df.shape}')
print(f'Total de colunas: {len(df.columns)}')
print('Colunas disponíveis:')
display(df.columns.to_list())